In [ ]:
import os, sys
import psycopg2

sys.path.append(os.path.abspath(os.path.join(os.path.dirname("__file__"), "..")))
from shared.embedding import embed
from _data_generator.pgvec_generator import generate


In [ ]:
conn = psycopg2.connect(
    host="pgvector_snoql_lab",
    database="vectordb",
    user="user",
    password="password"
)

cur = conn.cursor()

## Parametry Query

In [ ]:
query = "wireless headphones"
q_vec = embed([query])[0].tolist()

## 1. Podstawowe wyszukiwanie.

In [ ]:

cur.execute("""
SELECT name
FROM products
ORDER BY embedding <=> %s::vector
LIMIT 5
""", (q_vec,))

print(cur.fetchall())

## 2. Filtrowanie po kategoriach

In [ ]:
cur.execute("""
SELECT name, category
FROM products
WHERE category = 'electronics'
ORDER BY embedding <=> %s::vector
LIMIT 5
""", (q_vec,))

print(cur.fetchall())

## 3. Filtrowanie + Join

In [ ]:
cur.execute("""
SELECT p.name, p.price, s.sales_count
FROM products p
JOIN sales s ON p.id = s.product_id
WHERE p.price > 200
ORDER BY p.embedding <=> %s::vector
LIMIT 5
""", (q_vec,))

print(cur.fetchall())

## 4. Operator Cosine <=>

In [ ]:
cur.execute("""
SELECT name, embedding <=> %s::vector
FROM products
ORDER BY embedding <=> %s::vector
LIMIT 5
""", (q_vec, q_vec))

print(cur.fetchall())

## 5. Operator L2 <->

In [ ]:
cur.execute("""
SELECT name, embedding <-> %s::vector
FROM products
ORDER BY embedding <-> %s::vector
LIMIT 5
""", (q_vec, q_vec))

print(cur.fetchall())

## Pytania
1. Czy kolejność wyników jest taka sama?
2. Który operator lepiej działa dla tekstu?

## Wydajność

In [ ]:
cur.execute("""
EXPLAIN ANALYZE
SELECT *
FROM products
ORDER BY embedding <=> %s::vector
LIMIT 5
""", (q_vec,))
print("\n".join(r[0] for r in cur.fetchall()))

In [ ]:
cur.execute("DROP INDEX idx_products_embedding_cosine;")

In [ ]:
cur.execute("""
EXPLAIN ANALYZE
SELECT *
FROM products
ORDER BY embedding <=> %s::vector
LIMIT 5
""", (q_vec,))
print("\n".join(r[0] for r in cur.fetchall()))

## Pytania
1. Jak zmienił się plan wykonania?
2. Co oznacza Seq Scan?
3. Co oznacza Index Scan?

## Real example

In [ ]:
cur.execute("""
SELECT p.name, p.category, p.price, s.sales_count
FROM products p
JOIN sales s ON p.id = s.product_id
WHERE p.category = 'electronics'
AND p.price > 300
ORDER BY p.embedding <=> %s::vector
LIMIT 5
""", (q_vec,))

print(cur.fetchall())

## Pytania końcowe
1. Dlaczego pgvector jest bardziej elastyczny niż FAISS, czy Qdrant?
2. Jakie są koszty tej elastyczności?
Kiedy użyjesz:
- FAISS
- qdrant
- pgvector

In [ ]:
cur.close()
conn.close()